In [ ]:
import sys
import multiprocessing
import numpy as np 
import matplotlib.pyplot as plt
from T_method import LayeredStructure
from My_plotter import Style, Plotter
from scipy.optimize import minimize
from scipy.optimize import differential_evolution

In [ ]:
st = Style()
ndf = 150 # кол-во точек по df
min_df = -0.5
max_df = 0.5
df_width = (max_df - min_df)*100 # ширина по df в процентах
points_per_m_percent = lambda m: int(ndf * m / df_width) # кол-во точек на m процентов по df
df = np.linspace(min_df, max_df, ndf) # относительный сдвиг частоты
def sigma(x, thres=0, k=3):
    return 1/(1+np.exp(-k*(x - thres)))
def penalty(x, thres=0, k=3, c=3):
    return c/k * np.log(1 + np.exp(-k*(x - thres)))

In [ ]:
import pickle
import inspect

m_target = points_per_m_percent(25)  # 10% of df width

def objective_function(params):
    n = len(params) // 2
    
    alpha = np.array(params[:n])
    beta = np.array(params[n:])
    structure = LayeredStructure(alpha, beta=beta)
    directivity = 10*np.log10(structure.directivity(df))
    target_f = -np.inf
    for i in range(0, ndf-m_target):
        segment_i = directivity[i:i+m_target]
        target_f_i = np.sum(sigma(segment_i, thres=18, k=3) - penalty(segment_i, thres=18, k=3, c=0.1))
        if target_f_i > target_f:
            target_f = target_f_i
    print(f"Iteration {5}: target_f = {target_f/ndf*df_width:.4f}")
    return -target_f  # We want to maximize target_f, so we minimize the negative of it


bounds = [
    (0,  20.0), (-20.0, 0), (0, 20.0), (-20.0, 0), (0, 20.0),  # alpha
    (0.1,  7.0), ( 0.1, 1.0), ( 0.1, 1.0), ( 0.1, 1.0), ( 0.1, 1.0),  # beta
]


alpha0 = np.array([3, -1.2, 2.9, -1.2, 1])*1.4
beta0 = np.array([2.8, 0.75, 0.65, 0.6, 0.5])

initial_params = np.concatenate([alpha0, beta0])

print(objective_function(initial_params)/ndf*df_width)  # Check initial score

try:
    pickle.dumps(objective_function)
    print("✅ Функция сериализуется!")
except Exception as e:
    print(f"❌ Ошибка сериализации: {e}")
    print("\n🔍 Проверяем замыкания:")
    
    # Проверяем, какие глобальные переменные использует функция
    globals_used = inspect.getclosurevars(objective_function)
    print("Глобальные переменные:", globals_used.globals.keys())
    print("Неглобальные ссылки:", globals_used.nonlocals)

In [ ]:
# count = 0
# res = minimize(
#     objective_function,
#     x0=initial_params,
#     method="Powell",
#     bounds=bounds,  # SciPy will keep it within bounds for Powell
#     options=dict(
#         maxfev=2000,    # iterations of Powell outer loop
#         xtol=1e-3,      # param tolerance
#         ftol=1e-4,      # objective tolerance
#         disp=True
#     )
# )
# best_params = res.x
# best_score = -res.fun  # because we minimized -J

# print("Success:", res.success)
# print("Message:", res.message)
# print("Best score (sum of sigmoids):", best_score/ndf*df_width)
# print("Best params:", best_params)

In [ ]:
count = 0
if __name__ == "__main__":
    multiprocessing.freeze_support()

    res = differential_evolution(
        objective_function,
        bounds=bounds,
        strategy="best1bin",
        popsize=10,            # 10 * dim = 100 особей
        maxiter=40,            # можно увеличить позже
        workers=2,            # использовать все ядра
        updating="deferred",   # обязательно для параллелизма
        polish=False,           # polishing сделаем потом Powell
        seed=42
    )

    best_params = res.x
    best_score = -res.fun

    print("Best value:", best_score)
    print("Success:", res.success)
    print("Message:", res.message)
    print("Best score (sum of sigmoids):", best_score/ndf*df_width)
    print("Best params:", best_params)

Iteration 5: target_f = -109.7610
Iteration 5: target_f = -27.4926
Iteration 5: target_f = -91.5964
Iteration 5: target_f = -77.6868
Iteration 5: target_f = -114.3536
Iteration 5: target_f = -60.6614
Iteration 5: target_f = -38.1292
Iteration 5: target_f = -105.0971
Iteration 5: target_f = -169.1839
Iteration 5: target_f = -70.7211
Iteration 5: target_f = -63.3580
Iteration 5: target_f = -65.8787
Iteration 5: target_f = -106.9354
Iteration 5: target_f = -105.2261
Iteration 5: target_f = -94.9796

In [ ]:
alph_opt = np.array(best_params[:5])
beta_opt = np.array(best_params[5:])
alph_0 = np.array([3, -1.2, 2.9, -1.2, 1])*1.4
beta_0 = np.array([2.8, 0.75, 0.65, 0.6, 0.5])
structure_opt = LayeredStructure(alph_opt, beta=beta_opt)
structure_0 = LayeredStructure(alph_0, beta_0)
dir_prev = 10*np.log10(structure_0.directivity(df))
dir1 = 10*np.log10(structure_opt.directivity(df))
fig, ax = plt.subplots()
pl = Plotter(ax, st)
pl.plot(df*100, dir1, label='struct_0')
pl.plot(df*100, dir_prev, label='struct_opt')
pl.set_xlabel('df/f %')
pl.set_ylabel('Directivity (dB)')
pl.set_title('Directivity vs Frequency Offset')
pl.set_ylim((0, 25))
pl.finalize()
ax.axhline(18, color='gray', linestyle='--', alpha=0.5)
plt.show()

In [ ]:
print("Optimized alpha:", repr(best_params[:5]))
print("Optimized beta:", repr(best_params[5:]))